# Практика · Тема 10 · Правила й регулярні вирази

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

Ця практика друкує **всі** числа лекції. Головне питання теми одне:
**скільки розмічених прикладів коштує правило, написане за пʼять хвилин** — і де
проходить межа, за якою правило не допоможе взагалі.

Що зробимо:

1. зберемо задачу з чесною міткою: мітка з англійського оригіналу, ознаки з українського перекладу;
2. напишемо регулярний вираз на вісім патернів і заміряємо його точність і повноту;
3. побудуємо криву навчання логістичної регресії й **знайдемо точку перетину** з правилом;
4. покажемо, що ця точка залежить від того, з якою моделлю сперечаєшся;
5. знайдемо задачу, де правило дає рівно **1.0000**, а модель — ні;
6. заміряємо, як росте якість правила з кожним доданим патерном і де крива стає полицею;
7. покажемо задачу, де перебір патернів безнадійний;
8. зламаємо регулярний вираз катастрофічним відкатом і заміряємо час;
9. перевіримо гібрид: правило як ознака для моделі;
10. підсумуємо весь блок 2 одним замірoм на одній вибірці.

> ⏱ Зошит навчає близько пʼятдесяти моделей — переважно маленьких. Заміряно
> наодинці: **близько хвилини** на чотирьох ядрах без відеокарти. Якщо машина зайнята
> чимось іще, буде втричі довше.

In [ ]:
# ⚠️ Фіксуємо кількість потоків ДО імпорту numpy. Інакше BLAS розтягує роботу
# на всі ядра, і заміряний час перестає означати «скільки коштує ця модель»:
# на зайнятій машині процесорний час роздувається в десятки разів.
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

import sys, time, glob, gettext, collections, math
import re                      # стандартний модуль регулярних виразів
import regex as regex_module   # розширений: уміє рекурсію, \p{...} і присвійні квантифікатори
import numpy as np
import scipy.sparse as sparse
import sklearn

print("Python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("sklearn    ", sklearn.__version__)
print("regex      ", regex_module.__version__)

STARTED = time.time()
def elapsed(label):
    "Друкуємо, скільки часу з'їв кожен великий крок — бюджет зошита обмежений."
    print(f"   ⏱ {label}: {time.time() - STARTED:.0f} с від початку")

## 1 · Корпус

Той самий, що в усьому курсі: українські переклади інтерфейсів із `/usr/share/locale`.
У кожному записі лежить **англійський оригінал поруч з українським перекладом** — саме
це дає нам чесну мітку.

Якщо української локалі на машині немає, беремо вбудований мінікорпус. Числа на ньому
будуть інші, і зошит про це скаже прямо.

In [ ]:
def load_corpus():
    """Читає скомпільовані каталоги перекладів. Повертає (програма, оригінал, переклад)."""
    docs = []
    for path in sorted(glob.glob("/usr/share/locale/uk/LC_MESSAGES/*.mo")):
        try:
            with open(path, "rb") as handle:
                catalog = gettext.GNUTranslations(handle)
            program = path.split("/")[-1][:-3]
            for source, target in catalog._catalog.items():
                # беремо лише довгі рядки: короткі — це підписи кнопок, а не речення
                if isinstance(source, str) and isinstance(target, str) \
                   and len(target) > 30 and "Project-Id" not in target:
                    docs.append((program, source, target))
        except Exception:
            pass          # зіпсований або чужий формат каталогу — просто пропускаємо
    return docs


def mini_corpus():
    """Запасний варіант: якщо української локалі немає, збираємо корпус із заготовок."""
    heads_en = ["Cannot open", "Failed to read", "Unable to write", "Invalid value in",
                "Permission denied for", "No such entry in", "Could not parse",
                "Opening", "Reading", "Writing", "Checking", "Saving"]
    heads_uk = ["Не вдалося відкрити", "Помилка читання", "Неможливо записати",
                "Некоректне значення в", "Відмовлено в доступі до", "Не знайдено запис у",
                "Не вдалося розібрати", "Відкриваємо", "Читаємо", "Записуємо",
                "Перевіряємо", "Зберігаємо"]
    tails_en = ["configuration file %s", "the archive %d", "device --%s", "user profile",
                "the network share", "temporary directory", "the index of %s"]
    tails_uk = ["файл налаштувань %s", "архів %d", "пристрій --%s", "профіль користувача",
                "мережеву теку", "тимчасовий каталог", "каталог %s"]
    docs = []
    for i, (he, hu) in enumerate(zip(heads_en, heads_uk)):
        for j, (te, tu) in enumerate(zip(tails_en, tails_uk)):
            for k in range(6):
                suffix_en = f" (спроба {k})".replace("спроба", "attempt")
                docs.append((f"program{j}", f"{he} {te}{suffix_en}",
                             f"{hu} {tu} (спроба {k})"))
    return docs


corpus = load_corpus()
IS_REAL = len(corpus) > 10000
if not IS_REAL:
    print("⚠️  Української локалі не знайдено — працюємо на вбудованому мінікорпусі.")
    print("    Усі числа нижче будуть інші, ніж у лекції.")
    corpus = mini_corpus()
else:
    print("Локаль знайдено: /usr/share/locale/uk/LC_MESSAGES/")

programs = {program for program, _, _ in corpus}
print(f"документів {len(corpus)}, програм {len(programs)}")
print()
for program, source, target in corpus[:3]:
    print(f"  [{program}]")
    print(f"     EN: {source[:70]}")
    print(f"     UK: {target[:70]}")
elapsed("корпус зчитано")

## 2 · Задача з чесною міткою

Мітка береться з **англійського оригіналу**, ознаки — з **українського перекладу**.
Такий прийом уже вживала тема 03: модель не бачить того, з чого зроблено мітку, тож
замір не круговий.

Це важливо саме тут. Якби і мітку, і правило ми брали з українського боку, правило
просто списувало б відповідь, і його «якість» нічого не означала б.

In [ ]:
# мітка: чи є в англійському оригіналі слово про помилку
ERROR_IN_ENGLISH = re.compile(
    r"\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b", re.I)

labels = np.array([1 if ERROR_IN_ENGLISH.search(source) else 0
                   for _, source, _ in corpus])
texts = np.array([target for _, _, target in corpus], dtype=object)

print(f"документів            {len(texts)}")
print(f"частка «помилка»      {labels.mean():.4f}")
print(f"позитивів             {labels.sum()}")

lengths = [len(t.split()) for t in texts]
print(f"медіана довжини       {int(np.median(lengths))} слів")

## 3 · Правило: вісім патернів, пʼять хвилин роботи

Ось усе правило. Вертикальна риска `|` означає «або», `\w*` — «і будь-яке
продовження слова», `\b` — межа слова (щоб «помилка» не знаходилась усередині
іншого слова), `re.I` — не зважати на великі й малі літери.

Патерн `помилк\w*` покриває одразу «помилка», «помилки», «помилку», «помилкою» —
одна риска замість чотирьох. Для української це не дрібниця: слово живе в тексті
десятком форм.

In [ ]:
ERROR_IN_UKRAINIAN = re.compile(
    r"\b(помилк\w*|не вдалося|неможливо|збій|відмовлено|"
    r"не знайдено|некоректн\w*|недійсн\w*)\b", re.I)

rule_says = np.array([1 if ERROR_IN_UKRAINIAN.search(t) else 0 for t in texts])


def precision_recall_f1(true_labels, predicted):
    """Рахуємо руками, щоб було видно, з чого складається кожне число."""
    true_positive  = int(((predicted == 1) & (true_labels == 1)).sum())
    false_positive = int(((predicted == 1) & (true_labels == 0)).sum())
    false_negative = int(((predicted == 0) & (true_labels == 1)).sum())
    precision = true_positive / (true_positive + false_positive)
    recall    = true_positive / (true_positive + false_negative)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1, true_positive, false_positive, false_negative


p, r, f1, tp, fp, fn = precision_recall_f1(labels, rule_says)
print(f"правило на всьому корпусі:  P {p:.4f}   R {r:.4f}   F1 {f1:.4f}")
print(f"   вгадало {tp}, помилкових тривог {fp}, пропустило {fn}")

# перевіряємо, що бібліотека рахує те саме
from sklearn.metrics import precision_recall_fscore_support
p_lib, r_lib, f_lib, _ = precision_recall_fscore_support(labels, rule_says, average="binary")
assert np.allclose([p, r, f1], [p_lib, r_lib, f_lib]), "розрахунок розійшовся!"
print("✅ збігається з sklearn")

## 4 · Поділ на навчальну й перевірну частини

Правило нічого не вчить, тому йому поділ не потрібен. Але моделі потрібен, а
порівнювати їх треба **на тому самому** — тож і правило заміряємо на перевірній
частині теж.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"   # канонічний токенізатор блоку

positions = np.arange(len(texts))
train_idx, test_idx = train_test_split(
    positions, test_size=0.3, random_state=0, stratify=labels)
print(f"навчальна частина {len(train_idx)}, перевірна {len(test_idx)}")

p_test, r_test, f1_test, *_ = precision_recall_f1(labels[test_idx], rule_says[test_idx])
RULE_F1 = f1_test
print(f"правило на перевірній частині:  P {p_test:.4f}   R {r_test:.4f}   F1 {f1_test:.4f}")

## 5 · Крива навчання: скільки прикладів коштує правило

Тепер головний замір теми. Даємо логістичній регресії **N** розмічених прикладів і
дивимось, коли вона дожене правило.

Важлива деталь методики: словник TF-IDF будується **теж лише з цих N прикладів**.
Інакше ми б крадькома дали моделі знання про весь корпус, а питання ж стоїть інакше —
скільки треба **розмітити**, щоб зібрати модель із нуля.

Щоб не різати ті самі тексти на слова по п'ятдесят разів, ріжемо їх один раз наперед.

In [ ]:
split_into_words = TfidfVectorizer(lowercase=True,
                                   token_pattern=TOKEN_PATTERN).build_analyzer()
word_lists = [split_into_words(text) for text in texts]

def already_split(tokens):
    "Аналізатор, який нічого не робить: тексти вже нарізані на слова."
    return tokens

test_word_lists = [word_lists[i] for i in test_idx]
test_labels = labels[test_idx]
print(f"нарізано {len(word_lists)} документів")
elapsed("нарізка")

In [ ]:
def train_and_score(how_many, seed, class_weight=None, with_rule_column=False):
    """Навчаємо логістичну регресію на how_many прикладах і міряємо F1 на перевірній частині."""
    generator = np.random.default_rng(seed)
    if how_many >= len(train_idx):
        chosen = train_idx
    else:
        chosen = generator.choice(train_idx, size=how_many, replace=False)

    vectorizer = TfidfVectorizer(analyzer=already_split, min_df=2)
    train_matrix = vectorizer.fit_transform([word_lists[i] for i in chosen])
    test_matrix = vectorizer.transform(test_word_lists)

    if with_rule_column:
        # додаємо один стовпчик: спрацювала регулярка чи ні
        train_matrix = sparse.hstack([train_matrix, rule_says[chosen].reshape(-1, 1)]).tocsr()
        test_matrix = sparse.hstack([test_matrix, rule_says[test_idx].reshape(-1, 1)]).tocsr()

    model = LogisticRegression(max_iter=1000, class_weight=class_weight)
    model.fit(train_matrix, labels[chosen])
    return f1_score(test_labels, model.predict(test_matrix))


def learning_curve(sizes, class_weight=None, with_rule_column=False, title=""):
    """Три зерна на кожну точку: різниця, менша за розкид, не є різницею."""
    result = {}
    print(title)
    for size in sizes:
        scores = [train_and_score(size, seed, class_weight, with_rule_column)
                  for seed in (0, 1, 2)]
        spread = (max(scores) - min(scores)) / 2
        result[size] = scores
        print(f"   {size:>6} прикладів   F1 {np.mean(scores):.4f} ±{spread:.4f}"
              f"   ({' '.join(f'{s:.4f}' for s in scores)})")
    return result


COARSE = [200, 1000, 5000, 20000]
plain_curve = learning_curve(COARSE, title="проста логістична регресія:")
elapsed("груба крива")

Уже видно, де лежить відповідь: на 5 000 прикладах модель ще програє правилу, на
20 000 — уже виграє. Але «між пʼятьма й двадцятьма тисячами» — це не число.
Згустимо сітку.

In [ ]:
FINE = [6000, 7000, 8000, 9000, 10000]
plain_curve.update(learning_curve(FINE, title="дрібна сітка навколо перетину:"))
elapsed("дрібна сітка")

Тепер знайдемо точку перетину для **кожного зерна окремо** й подивимось на розкид.
Між двома сусідніми точками сітки крива йде майже прямою в логарифмі кількості
прикладів — там і інтерполюємо.

In [ ]:
def crossing_point(curve, threshold):
    """Скільки прикладів треба, щоб F1 моделі дорівняла порогу. Окремо на кожному зерні."""
    sizes = sorted(curve)
    answers = []
    for seed_index in range(3):
        previous = None
        for size in sizes:
            score = curve[size][seed_index]
            if previous is not None and previous[1] < threshold <= score:
                left_size, left_score = previous
                # частка шляху від лівої точки до правої, на якій крива перетинає поріг
                share = (threshold - left_score) / (score - left_score)
                answers.append(math.exp(math.log(left_size)
                                        + share * (math.log(size) - math.log(left_size))))
                break
            previous = (size, score)
    return answers


crossings = crossing_point(plain_curve, RULE_F1)
CROSS_MEAN = np.mean(crossings)
CROSS_SPREAD = (max(crossings) - min(crossings)) / 2
print(f"поріг — F1 правила: {RULE_F1:.4f}")
print(f"перетин по зернах:  {'  '.join(f'{c:.0f}' for c in crossings)}")
print()
print(f"➜ ВІСІМ ПАТЕРНІВ КОШТУЮТЬ {CROSS_MEAN:.0f} ±{CROSS_SPREAD:.0f} РОЗМІЧЕНИХ ПРИКЛАДІВ")

І друга половина відповіді: а що модель має **на всіх** даних? Це верхня межа, до якої
правилу вже не дотягнутись.

In [ ]:
# на всій навчальній частині зерно нічого не міняє: вибірка одна й та сама
full_score = train_and_score(len(train_idx), 0)
print(f"модель на всіх {len(train_idx)} прикладах:  F1 {full_score:.4f}")
print(f"правило:                            F1 {RULE_F1:.4f}")
print(f"перевага моделі на всіх даних:      {full_score - RULE_F1:+.4f}")

# те саме зерно й ті самі дані мусять дати те саме число (грабля №28)
assert train_and_score(len(train_idx), 1) == full_score, "модель невідтворювана!"
print("✅ два прогони на тих самих даних дають те саме число")
elapsed("уся навчальна частина")

## 6 · Число залежить від того, з якою моделлю сперечаєшся

Тема 06 заміряла, що `class_weight='balanced'` на цій самій задачі дає і кращу якість,
і швидшу збіжність. Логічно спитати: а що станеться з точкою перетину, якщо взяти не
логістичну регресію за замовчуванням, а налаштовану?

In [ ]:
BALANCED = [200, 400, 700, 1000, 2000, 5000, 20000]
balanced_curve = learning_curve(BALANCED, class_weight="balanced",
                                title="логістична з class_weight='balanced':")

balanced_crossings = crossing_point(balanced_curve, RULE_F1)
BAL_MEAN = np.mean(balanced_crossings)
BAL_SPREAD = (max(balanced_crossings) - min(balanced_crossings)) / 2
print()
print(f"перетин по зернах:  {'  '.join(f'{c:.0f}' for c in balanced_crossings)}")
print(f"➜ ТА САМА РЕГУЛЯРКА КОШТУЄ {BAL_MEAN:.0f} ±{BAL_SPREAD:.0f} ПРИКЛАДІВ")
print(f"   різниця з простою моделлю: у {CROSS_MEAN / BAL_MEAN:.1f} раза")
balanced_full = train_and_score(len(train_idx), 0, "balanced")
print(f"   balanced на всіх {len(train_idx)} прикладах: F1 {balanced_full:.4f}")
elapsed("balanced")

## 7 · Гібрид: правило як ознака для моделі

Найпрактичніше застосування правила — не заміняти ним модель, а **віддати його моделі
як ще один стовпчик**. Один біт: спрацювала регулярка чи ні.

In [ ]:
print("гібрид: TF-IDF + один стовпчик «регулярка спрацювала»")
for size in COARSE:
    hybrid_scores = [train_and_score(size, seed, None, True) for seed in (0, 1, 2)]
    plain_mean = np.mean(plain_curve[size])
    spread = (max(hybrid_scores) - min(hybrid_scores)) / 2
    print(f"   {size:>6}   гібрид {np.mean(hybrid_scores):.4f} ±{spread:.4f}"
          f"   проста {plain_mean:.4f}   різниця {np.mean(hybrid_scores) - plain_mean:+.4f}")

hybrid_full = train_and_score(len(train_idx), 0, None, True)
print(f"   {len(train_idx):>6}   гібрид {hybrid_full:.4f}"
      f"                   проста {full_score:.4f}   різниця {hybrid_full - full_score:+.4f}")
elapsed("гібрид")

## 8 · Де правило виграє назавжди

Досі ми міряли задачу, у якій правило рано чи пізно програє. Тепер — задача, у якій
воно не програє **ніколи**: специфікатор формату `printf`. Це `%s`, `%d`, `%.2f`,
`%3$-.4ld` — шматочки, куди програма підставляє значення.

Ця мова описана стандартом C, а не звичкою людей. Там, де мова формальна, правило не
наближається до відповіді — воно **і є** відповіддю.

Щоб довести це чесно, потрібні **дві незалежні реалізації** тієї самої граматики.
Спершу — посимвольний розбирач, у якому немає жодного регулярного виразу.

In [ ]:
FLAG_CHARACTERS = "-+ #0'"
LENGTH_MODIFIERS = ("hh", "ll", "h", "l", "L", "q", "j", "z", "t")
CONVERSION_CHARACTERS = "diouxXeEfFgGaAcspn%"


def scan_format_specifiers(text):
    """Знаходить специфікатори printf, читаючи рядок символ за символом.

    Порядок частин у стандарті такий:
      % [номер$] [прапорці] [ширина] [.точність] [довжина] буква
    """
    found = []
    i = 0
    size = len(text)
    while i < size:
        if text[i] != "%":
            i += 1
            continue
        j = i + 1
        # необовʼязковий номер аргументу: цифри, за якими стоїть долар
        k = j
        while k < size and text[k].isdigit():
            k += 1
        if k > j and k < size and text[k] == "$":
            j = k + 1
        while j < size and text[j] in FLAG_CHARACTERS:
            j += 1
        while j < size and text[j].isdigit():        # ширина поля
            j += 1
        if j < size and text[j] == ".":              # точність (цифри необовʼязкові)
            j += 1
            while j < size and text[j].isdigit():
                j += 1
        for modifier in LENGTH_MODIFIERS:
            if text.startswith(modifier, j):
                j += len(modifier)
                break
        if j < size and text[j] in CONVERSION_CHARACTERS:
            found.append(text[i:j + 1])
            i = j + 1
        else:
            i += 1                                    # це був просто відсоток
    return found


print(scan_format_specifiers("Готово %3$-.4ld із %d файлів, %.1f%%"))

Тепер той самий стандарт, записаний одним регулярним виразом. І одразу — звірка:
розбирач і регулярка мусять дати **однакові** списки на всіх рядках корпусу, і
англійських, і українських.

In [ ]:
SPECIFIER_FIRST_TRY = re.compile(
    r"%(?:\d+\$)?[-+ #0']*\d*(?:\.\d+)?(?:hh|ll|[hlLqjzt])?[diouxXeEfFgGaAcspn%]")

all_strings = [target for _, _, target in corpus] + [source for _, source, _ in corpus]

disagreements = []
for line in all_strings:
    if scan_format_specifiers(line) != SPECIFIER_FIRST_TRY.findall(line):
        disagreements.append(line)

print(f"рядків звірено: {len(all_strings)}")
print(f"розбіжностей:   {len(disagreements)}")
for line in disagreements[:2]:
    print(f"   {line!r}")
    print(f"      розбирач: {scan_format_specifiers(line)}")
    print(f"      регулярка: {SPECIFIER_FIRST_TRY.findall(line)}")

Ось заради чого потрібні дві реалізації. Розбіжність знайшлась — і **помилялась
регулярка**: у стандарті C точність можна писати без цифр (`%.f` означає «точність
нуль»), а `(?:\.\d+)?` вимагає хоча б одну цифру. Виправляємо `\d+` на `\d*`.

Це і є чесний спосіб дійти до одиниці: не оголосити її, а знайти незгоду й розібратись,
хто правий.

In [ ]:
SPECIFIER = re.compile(
    r"%(?:\d+\$)?[-+ #0']*\d*(?:\.\d*)?(?:hh|ll|[hlLqjzt])?[diouxXeEfFgGaAcspn%]")

mismatches = sum(1 for line in all_strings
                 if scan_format_specifiers(line) != SPECIFIER.findall(line))
print(f"розбіжностей після виправлення: {mismatches} із {len(all_strings)}")
assert mismatches == 0, "дві реалізації однієї граматики мусять збігатися!"
print("✅ дві незалежні реалізації збігаються символ у символ")

format_labels = np.array([1 if scan_format_specifiers(t) else 0 for t in texts])
print(f"документів зі специфікатором: {format_labels.sum()} ({format_labels.mean():.4f})")
elapsed("розбір формату")

Тепер поставимо ту саму задачу моделі. Мітку дає розбирач, ознаки — сам текст. Щоб
дати моделі чесний шанс, беремо **символьні** n-грами: словесний токенізатор курсу
пропускає латиницю й відсотки, і модель просто не побачила б `%s`.

In [ ]:
format_train, format_test = train_test_split(
    positions, test_size=0.3, random_state=0, stratify=format_labels)
format_test_texts = list(texts[format_test])
format_test_labels = format_labels[format_test]

rule_on_format = np.array([1 if SPECIFIER.search(t) else 0 for t in texts])
p_fmt, r_fmt, f1_fmt, *_ = precision_recall_f1(format_test_labels, rule_on_format[format_test])
print(f"правило на форматі:  P {p_fmt:.4f}   R {r_fmt:.4f}   F1 {f1_fmt:.4f}")

format_scores = []
for seed in (0, 1, 2):
    generator = np.random.default_rng(seed)
    chosen = generator.choice(format_train, size=10000, replace=False)
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3),
                                      min_df=3, max_features=6000, lowercase=True)
    train_matrix = char_vectorizer.fit_transform(list(texts[chosen]))
    test_matrix = char_vectorizer.transform(format_test_texts)
    char_model = LogisticRegression(max_iter=1000).fit(train_matrix, format_labels[chosen])
    predicted = char_model.predict(test_matrix)
    format_scores.append(f1_score(format_test_labels, predicted))
    if seed == 0:
        p_m, r_m, f_m, *_ = precision_recall_f1(format_test_labels, predicted)
        wrong = int((predicted != format_test_labels).sum())
        print(f"модель на 10 000:    P {p_m:.4f}   R {r_m:.4f}   F1 {f_m:.4f}")
        print(f"   помилок: {wrong} із {len(format_test_labels)}")

spread = (max(format_scores) - min(format_scores)) / 2
print(f"модель, три зерна:   F1 {np.mean(format_scores):.4f} ±{spread:.4f}")
elapsed("модель на форматі")

Число 0.95 виглядає пристойно, поки не спитати, **де саме** ці пʼять відсотків. Ось
пʼять специфікаторів, яких у корпусі немає жодного разу. Правило їх знає, бо знає
граматику. Модель знає лише те, що бачила.

In [ ]:
# перші два рядки взяті з корпусу, решта — вигадані
examples = ["Помилка під час розпізнавання: код 0x%02x",
            "Не вдалося встановити gid %ld: %s",
            "Готово %3$-.4ld файлів",
            "Залишилось %+08.3Lf секунд",
            "Код помилки %#jx",
            "Обробка %.f%% завершена"]

print(f"{'рядок':<40} {'разів у корпусі':>15} {'правило':<12} модель")
for line in examples:
    specifiers = scan_format_specifiers(line)
    by_rule = " ".join(SPECIFIER.findall(line))
    by_model = int(char_model.predict(char_vectorizer.transform([line]))[0])
    # скільки разів точнісінько такий набір специфікаторів є в корпусі
    times = sum(1 for other in all_strings
                if all(s in other for s in specifiers))
    assert specifiers == SPECIFIER.findall(line), "розбирач і правило розійшлися!"
    print(f"{line:<40} {times:>15} {by_rule:<12} {by_model}")

## 9 · Ціна підтримки: як росте якість із кожним патерном

Правило треба доповнювати — це його справжня ціна. Заміряємо її чесно: хай **машина**
добирає патерни жадібно, по одному, щоразу беручи той, який найбільше піднімає F1 на
навчальній частині. Так ми побачимо найкращу можливу криву, а не криву нашої
кмітливості.

Кандидатів беремо двох сортів:

* **словоформи** — «некоректний», «некоректна», «некоректне» — це різні патерни;
* **основи** — спільний початок кількох словоформ, тобто те саме, що `некоректн\w*`.

Порівняння цих двох кривих і є відповіддю на питання «скільки коштує українська
морфологія в патернах».

In [ ]:
presence = CountVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN,
                           binary=True, min_df=20).fit(texts)
matrix = presence.transform(texts).tocsc()
vocabulary = list(presence.get_feature_names_out())

# для кожного слова — список документів, де воно є
documents_with_word = {word: matrix.indices[matrix.indptr[j]:matrix.indptr[j + 1]]
                       for j, word in enumerate(vocabulary)}

# кандидати-основи: кожен початок слова довжиною від чотирьох літер
by_stem = {}
for word, rows in documents_with_word.items():
    for length in range(4, len(word) + 1):
        by_stem.setdefault(word[:length], []).append(rows)
documents_with_stem = {}
for stem, parts in by_stem.items():
    if len(parts) == 1:
        documents_with_stem[stem] = parts[0]
    else:
        documents_with_stem[stem] = np.unique(np.concatenate(parts))

print(f"кандидатів-словоформ: {len(documents_with_word)}")
print(f"кандидатів-основ:     {len(documents_with_stem)}")
elapsed("кандидати")

In [ ]:
is_train = np.zeros(len(texts), bool); is_train[train_idx] = True
is_test = np.zeros(len(texts), bool); is_test[test_idx] = True
positives_train = int(labels[is_train].sum())
positives_test = int(labels[is_test].sum())


def greedy_patterns(candidates, max_steps, title):
    """Жадібно додаємо патерни: щоразу той, що найбільше піднімає F1 на навчальній частині.

    Правило — це логічне АБО патернів, тож новий патерн може лише додати документів,
    ніколи не забрати. Тому рахуємо тільки те, що він додає понад уже покрите.
    """
    covered = np.zeros(len(texts), bool)
    true_pos = false_pos = 0
    true_pos_test = false_pos_test = 0
    remaining = list(candidates)
    train_rows = {name: candidates[name][is_train[candidates[name]]] for name in remaining}
    history = []
    print(title)
    for step in range(max_steps):
        best = None
        for name in remaining:
            rows = train_rows[name]
            fresh = rows[~covered[rows]]              # ще не покриті цим правилом
            added_right = int(labels[fresh].sum())
            added_wrong = len(fresh) - added_right
            tp = true_pos + added_right
            fp = false_pos + added_wrong
            score = 2 * tp / (2 * tp + fp + (positives_train - tp)) if tp else 0.0
            if best is None or score > best[0]:
                best = (score, name, added_right, added_wrong)
        score, name, added_right, added_wrong = best
        if history and score <= history[-1][1] + 1e-9:
            print(f"   ПОЛИЦЯ: після {len(history)} патернів жоден наступний уже не додає F1")
            break
        true_pos += added_right
        false_pos += added_wrong
        rows_all = candidates[name]
        fresh_all = rows_all[~covered[rows_all]]
        fresh_test = fresh_all[is_test[fresh_all]]
        added_right_test = int(labels[fresh_test].sum())
        true_pos_test += added_right_test
        false_pos_test += len(fresh_test) - added_right_test
        covered[rows_all] = True
        remaining.remove(name)
        f1_test_now = 2 * true_pos_test / (2 * true_pos_test + false_pos_test
                                           + (positives_test - true_pos_test))
        history.append((name, score, f1_test_now))
        print(f"   {step + 1:>2}  +{name:<14}  train {score:.4f}   test {f1_test_now:.4f}")
    precision_now = true_pos_test / (true_pos_test + false_pos_test)
    recall_now = true_pos_test / positives_test
    missed = int((~covered & is_test & (labels == 1)).sum())
    return history, precision_now, recall_now, missed


stem_history, stem_p, stem_r, stem_missed = greedy_patterns(
    documents_with_stem, 30, "жадібний добір основ:")
print()
print(f"стеля правил: {len(stem_history)} патернів, "
      f"test F1 {stem_history[-1][2]:.4f}, P {stem_p:.4f}, R {stem_r:.4f}")
print(f"непокритих позитивів у перевірній частині: {stem_missed} із {positives_test}"
      f" ({stem_missed / positives_test:.4f})")
elapsed("основи")

In [ ]:
word_history, *_ = greedy_patterns(documents_with_word, 20, "жадібний добір словоформ:")
print()
print(f"вісім основ:     test F1 {stem_history[7][2]:.4f}")
print(f"вісім словоформ: test F1 {word_history[7][2]:.4f}")
print(f"двадцять словоформ: test F1 {word_history[19][2]:.4f}")
print()
print("➜ вісім основ дають те саме, що двадцять словоформ — це й є ціна морфології")
elapsed("словоформи")

І ще одне число, яке варто побачити поруч: **стеля правил проти моделі на всіх даних**.
Жадібний добір мав доступ до всіх 65 374 міток — тобто це найкраще правило, яке взагалі
можна скласти з таких патернів, і воно все одно нижче за модель.

In [ ]:
print(f"стеля правил (23 патерни, усі мітки видно):  {stem_history[-1][2]:.4f}")
print(f"модель на всіх {len(train_idx)} прикладах:              {full_score:.4f}")
print(f"різниця:                                     {full_score - stem_history[-1][2]:+.4f}")
print()
print(f"а правило з восьми патернів, написане без жодної мітки:  {RULE_F1:.4f}")

## 10 · Де правило безсиле

Стеля з попереднього розділу — це не лінь автора правил. Це властивість самого підходу:
правило порівнює **написання**, а не зміст.

Найкоротший доказ уже лежить у нашому корпусі. Той самий англійський рядок різні
перекладачі переклали по-різному — і ми маємо пари документів, про які **точно**
відомо, що вони означають одне й те саме. Тема 05 брала саме ці пари, щоб зламати
TF-IDF; відтворимо її вибірку тут, щоб числа сходились.

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")

def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка з малим перекриттям."""
    by_source = collections.defaultdict(dict)
    for program, source, target in corpus:
        if source.strip() == "translator-credits":
            continue
        by_source[source][" ".join(target.split())] = program
    found = []
    for source, variants in by_source.items():
        variant_texts = list(variants)
        for i in range(len(variant_texts)):
            for j in range(i + 1, len(variant_texts)):
                first, second = variant_texts[i], variant_texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(split_into_words(first))
                b = set(split_into_words(second))
                if len(a) < 3 or len(b) < 3:
                    continue
                if len(a & b) / len(a | b) < 0.34:
                    found.append((first, second, len(a & b)))
    return found


pairs = paraphrase_pairs()
without_common_word = sum(1 for pair in pairs if pair[2] == 0)
print(f"пар «те саме іншими словами»: {len(pairs)}")
print(f"з них без жодного спільного слова: {without_common_word}"
      f" ({without_common_word / len(pairs):.4f})")
# а якщо порівнювати не слова, а їхні перші чотири літери — тобто корені?
def roots(text):
    return {word[:4] for word in split_into_words(text)}

without_common_root = sum(1 for first, second, _ in pairs if not (roots(first) & roots(second)))
print(f"з них без жодного спільного кореня: {without_common_root}"
      f" ({without_common_root / len(pairs):.4f})")

# контроль: а скільки спільного мають випадкові пари, які нічого не означають разом?
generator = np.random.default_rng(0)
random_word = random_root = 0
for _ in range(len(pairs)):
    i, j = generator.integers(0, len(texts), 2)
    if set(split_into_words(texts[i])) & set(split_into_words(texts[j])):
        random_word += 1
    if roots(texts[i]) & roots(texts[j]):
        random_root += 1
print(f"випадкові пари: спільне слово в {random_word / len(pairs):.4f}, "
      f"спільний корінь у {random_root / len(pairs):.4f}")
print(f"правило «спільне слово» ловить {1 - without_common_word / len(pairs):.4f} пар змісту, "
      f"правило «спільний корінь» — {1 - without_common_root / len(pairs):.4f}")

# а тепер найважливіше: що буде, якщо шукати такі пари в усьому корпусі
all_pairs = len(texts) * (len(texts) - 1) // 2
false_rate = random_root / len(pairs)
print(f"пар документів у корпусі всього: {all_pairs}")
print(f"правило «спільний корінь» відповіло б «те саме» на {int(all_pairs * false_rate)} пар,")
print(f"   а справжніх серед них — {len(pairs)}; точність правила {len(pairs) / (all_pairs * false_rate):.9f}")
print()
print("пари, у яких немає навіть спільного кореня — і правило до них не дотягнеться:")
for first, second, _ in pairs:
    if not (roots(first) & roots(second)):
        print(f"    A: {first[:76]}")
        print(f"    B: {second[:76]}")
        print()

Ось межа, і вона не рухається. Щоб правило впоралося з такою парою, треба **окремий
патерн на кожну пару слів**, яка означає те саме. Скільки їх у мові — ніхто не рахував,
але точно не двадцять три.

Тема 05 заміряла цю ж біду з іншого боку: косинус TF-IDF на цих парах — **0.2230**,
тобто рівно стільки ж, скільки в двох випадкових документів. Відповідь на це питання
дає не правило й не мішок слів, а блок 3 курсу.

## 11 · Катастрофічний відкат

Останнє, що треба знати про регулярні вирази, — як ними покласти сервіс.

Патерн `^(\w+\s?)+$` виглядає невинно: «слова, розділені необовʼязковими пробілами».
Але всередині нього повтор укладено в повтор. Коли рядок **не** підходить, рушій мусить
перебрати всі способи розділити його на групи — а їх експоненційно багато.

In [ ]:
NESTED_REPEAT = re.compile(r"^(\w+\s?)+$")


def best_of_three(pattern, probe):
    """Беремо найкращий із трьох прогонів: сусідні процеси псують заміри часу."""
    times = []
    for _ in range(3):
        started = time.perf_counter()
        pattern.match(probe)
        times.append(time.perf_counter() - started)
    return min(times)


print("рядок «aaaa…a!» — не підходить під патерн, і рушій це доводить перебором")
print(f"{'довжина':>8} {'час, с':>12} {'у скільки разів більше':>24}")
previous = None
backtracking = []
for length in range(14, 25):
    probe = "a" * length + "!"
    took = best_of_three(NESTED_REPEAT, probe)
    ratio = f"×{took / previous:.2f}" if previous else "—"
    print(f"{length:>8} {took:>12.6f} {ratio:>24}")
    backtracking.append((length, took))
    previous = took

# кожна літера подвоює час: перевіряємо це середнім відношенням
ratios = [b / a for (_, a), (_, b) in zip(backtracking, backtracking[1:])]
print(f"середнє відношення сусідніх замірів: {np.mean(ratios):.2f}")

Кожна додана літера **подвоює** час. Це не «повільно» — це інший клас складності.
Тридцять літер уже дали б години.

Лікується двома способами, і обидва дешеві.

In [ ]:
probe = "a" * 24 + "!"

# 1) переписати так, щоб повтор не вкладався в повтор
NO_NESTING = re.compile(r"^\w+(?:\s\w+)*$")
NO_NESTING.match(probe)          # холостий прогін: перший виклик платить за розігрів
started = time.perf_counter()
for _ in range(1000):
    NO_NESTING.match(probe)
rewritten = (time.perf_counter() - started) / 1000

# 2) присвійний квантифікатор ++ — «взяв і не віддаю назад».
#    У стандартному re його немає, у модулі regex є.
POSSESSIVE = regex_module.compile(r"^(\w+\s?)++$")
POSSESSIVE.match(probe)          # те саме: модуль regex перший виклик робить довше
started = time.perf_counter()
for _ in range(1000):
    POSSESSIVE.match(probe)
possessive = (time.perf_counter() - started) / 1000

naive = best_of_three(NESTED_REPEAT, probe)

print(f"вкладений повтор, 25 символів:   {naive:.6f} с")
print(f"переписаний без вкладення:       {rewritten:.8f} с")
print(f"присвійний ++ із модуля regex:   {possessive:.8f} с")
print(f"пришвидшення переписуванням:     у {naive / rewritten:.0f} разів")
assert NO_NESTING.match("не вдалося відкрити файл"), "переписаний патерн має працювати так само"
assert not NO_NESTING.match(probe), "і так само відхиляти те, що відхиляв"
print("✅ переписаний патерн приймає ті самі рядки й відхиляє ті самі")
elapsed("відкат")

## 12 · Підсумок блоку 2: пʼять поглядів на одну задачу

Блок 2 — це пʼять способів щось зробити з текстом. Проженемо їх усі через **одну й ту
саму** вибірку на 20 000 документів і подивимось, як вони складаються в одну лінію.

In [ ]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.decomposition import NMF
from sklearn.metrics import precision_recall_fscore_support

generator = np.random.default_rng(0)
sample = generator.choice(train_idx, size=20000, replace=False)
vectorizer = TfidfVectorizer(analyzer=already_split, min_df=2)
sample_matrix = vectorizer.fit_transform([word_lists[i] for i in sample])
test_matrix = vectorizer.transform(test_word_lists)
print(f"вибірка {len(sample)}, ознак {sample_matrix.shape[1]}")

print("\n06 · класифікувати")
for name, make in (("MultinomialNB", MultinomialNB), ("ComplementNB", ComplementNB)):
    started = time.process_time()
    model = make().fit(sample_matrix, labels[sample])
    took = time.process_time() - started
    print(f"   {name:<20} F1 {f1_score(test_labels, model.predict(test_matrix)):.4f}"
          f"   процесорний час {took:.3f} с")
started = time.process_time()
logistic = LogisticRegression(max_iter=1000).fit(sample_matrix, labels[sample])
logistic_time = time.process_time() - started
predicted = logistic.predict(test_matrix)
print(f"   {'LogisticRegression':<20} F1 {f1_score(test_labels, predicted):.4f}"
      f"   процесорний час {logistic_time:.2f} с")

In [ ]:
print("07 · чесно оцінити")
accuracy = accuracy_score(test_labels, predicted)
_, _, micro_f1, _ = precision_recall_fscore_support(test_labels, predicted, average="micro")
_, _, macro_f1, _ = precision_recall_fscore_support(test_labels, predicted, average="macro")
print(f"   точність {accuracy:.4f}   мікро-F1 {micro_f1:.4f}   макро-F1 {macro_f1:.4f}")
assert abs(accuracy - micro_f1) < 1e-12, "мікро-F1 мусить дорівнювати точності!"
print("   ✅ мікро-F1 тотожно дорівнює точності — це та сама метрика під іншою назвою")

always_zero = np.zeros_like(test_labels)
_, _, dumb_macro, _ = precision_recall_fscore_support(test_labels, always_zero,
                                                     average="macro", zero_division=0)
print(f"   модель «усе — не помилка»: точність {accuracy_score(test_labels, always_zero):.4f},"
      f" макро-F1 {dumb_macro:.4f}")

In [ ]:
print("08 · знайти структуру без міток")
counts = CountVectorizer(analyzer=already_split, min_df=5, max_df=0.5)
count_matrix = counts.fit_transform([word_lists[i] for i in sample])
started = time.time()
topics = NMF(n_components=8, random_state=0, init="nndsvda", max_iter=300).fit(count_matrix)
print(f"   NMF на вісім тем за {time.time() - started:.1f} с")
weights = topics.transform(count_matrix)
names = np.array(counts.get_feature_names_out())
best_topic = None
for k in range(8):
    correlation = np.corrcoef(weights[:, k], labels[sample])[0, 1]
    if best_topic is None or correlation > best_topic[0]:
        best_topic = (correlation, k, " ".join(names[np.argsort(-topics.components_[k])[:6]]))
print(f"   найближча до мітки тема {best_topic[1]}: кореляція {best_topic[0]:+.4f}")
print(f"      топ-слова: {best_topic[2]}")
print("   ➜ не бачивши жодної мітки, NMF назвала ті самі слова, що й наше правило")

In [ ]:
print("09 · шукати")
# BM25 своїми руками: те саме правило, подане як запит із ранжуванням
test_counts = CountVectorizer(analyzer=already_split, min_df=2)
test_count_matrix = test_counts.fit_transform(test_word_lists).tocsr()
document_frequency = np.asarray((test_count_matrix > 0).sum(axis=0)).ravel()
total = test_count_matrix.shape[0]
inverse_frequency = np.log(1 + (total - document_frequency + 0.5) / (document_frequency + 0.5))
document_length = np.asarray(test_count_matrix.sum(axis=1)).ravel()
average_length = document_length.mean()
k1, b = 1.5, 0.75

entries = test_count_matrix.tocoo()
saturated = (inverse_frequency[entries.col] * entries.data * (k1 + 1)
             / (entries.data + k1 * (1 - b + b * document_length[entries.row] / average_length)))
bm25 = sparse.csr_matrix((saturated, (entries.row, entries.col)),
                         shape=test_count_matrix.shape)

query = ["помилка", "вдалося", "неможливо", "збій", "відмовлено",
         "знайдено", "некоректний", "недійсний"]
query_columns = [test_counts.vocabulary_[w] for w in query if w in test_counts.vocabulary_]
scores = np.asarray(bm25[:, query_columns].sum(axis=1)).ravel()

flagged = int(rule_says[test_idx].sum())
ranked = np.argsort(-scores)[:flagged]
hit = int(test_labels[ranked].sum())
precision_bm25 = hit / flagged
recall_bm25 = hit / test_labels.sum()
f1_bm25 = 2 * precision_bm25 * recall_bm25 / (precision_bm25 + recall_bm25)
print(f"   ті самі вісім слів як запит BM25, відсічення на {flagged} документах:")
print(f"      P {precision_bm25:.4f}   R {recall_bm25:.4f}   F1 {f1_bm25:.4f}")
print(f"   правило на тих самих словах:  P {p_test:.4f}   R {r_test:.4f}   F1 {RULE_F1:.4f}")
print("   ➜ ранжування дає ручку, а не якість: точку відсічення можна рухати, "
      "але в цій задачі правило точніше")

In [ ]:
print("10 · зрозуміти, коли моделі не треба")
print(f"   правило з восьми патернів, нуль розмічених прикладів:  F1 {RULE_F1:.4f}")
print(f"   модель, щоб дорівняти йому, потребує {CROSS_MEAN:.0f} ±{CROSS_SPREAD:.0f} прикладів")
print(f"   з class_weight='balanced' — уже {BAL_MEAN:.0f} ±{BAL_SPREAD:.0f}")
print(f"   модель на всіх {len(train_idx)}:  F1 {full_score:.4f}  ({full_score - RULE_F1:+.4f})")
print(f"   гібрид на всіх {len(train_idx)}:  F1 {hybrid_full:.4f}  ({hybrid_full - full_score:+.4f})")
print(f"   формальна мова (printf): правило {f1_fmt:.4f}, модель {np.mean(format_scores):.4f}")
print()
print(f"Усього: {time.time() - STARTED:.0f} с")

## Завдання

### 🟢 Рівень 1
Додай до правила один свій патерн — наприклад, `\bне вдається\b` або `\bвідмов\w*\b`.
Заміряй P, R і F1 до і після. **Зроблено, якщо** ти можеш сказати, за що саме заплатив:
скільки повнота виросла й скільки точність упала.

### 🟡 Рівень 2
Побудуй криву перетину для `ComplementNB` замість логістичної регресії. Наївний Баєс
учиться в сотні разів швидше — чи потрібно йому більше прикладів, щоб дорівняти
правилу? **Зроблено, якщо** названо точку перетину з розкидом по трьох зернах і
сказано, чи вона відрізняється від 7 616 більше, ніж на розкид.

### 🔴 Рівень 3
Побудуй правило на **регулярних виразах із групами**, а не на списку слів: наприклад,
`(не|немає|бракує)\s+\w*(вдал|можл|прав)` — і прожени через нього жадібний добір.
**Зроблено, якщо** побудовано криву «F1 від кількості патернів» для такої сімʼї
й показано, вища чи нижча в неї полиця, ніж 0.8799 у сімʼї простих основ.